In [22]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import r2_score,accuracy_score,classification_report,confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [9]:
df=pd.read_csv(r"D:\ML\datasets\retitanicdata.csv")

In [10]:
df.head(1)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,NaN,S


In [11]:
x=df.drop(columns=['Survived','PassengerId','Name','Ticket'])
y=df['Survived']

In [18]:
obj_num=x.select_dtypes(include='object').columns
x[obj_num]=x[obj_num].astype('category')

In [19]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   Pclass    891 non-null    int64   
 1   Sex       891 non-null    category
 2   Age       714 non-null    float64 
 3   SibSp     891 non-null    int64   
 4   Parch     891 non-null    int64   
 5   Fare      891 non-null    float64 
 6   Cabin     204 non-null    category
 7   Embarked  889 non-null    category
dtypes: category(3), float64(2), int64(3)
memory usage: 43.9 KB


In [20]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [23]:
num_col=xtrain.select_dtypes(exclude='category').columns
cat_col=xtrain.select_dtypes(include='number').columns


In [21]:
num_pipe=Pipeline([
    ('imputer',SimpleImputer(strategy='median'))
])
cat_pipe=Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoding',OneHotEncoder(handle_unknown='ignore'))
])

In [24]:
preprocessor=ColumnTransformer([
    ('num',num_pipe,num_col),
    ('cat',cat_pipe,cat_col)
])

In [25]:
model=Pipeline(
    [
        ('preprocessing',preprocessor),
        ('model',XGBClassifier(random_state=42,eval_metrics='logloss'))
    ]
)

In [26]:
model.fit(xtrain,ytrain)

c:\Users\Aditya Sarapure\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:17:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "eval_metrics" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [27]:
train_pred=model.predict(xtrain)
test_pred=model.predict(xtest)

In [28]:
print("Training Accuracy:",accuracy_score(ytrain,train_pred))
print("Testing Accuracy:",accuracy_score(ytest,test_pred))

Training Accuracy: 0.9171348314606742
Testing Accuracy: 0.7039106145251397
